<a href="https://colab.research.google.com/github/txellbalada/Reto_IA/blob/main/Reto_Telefonica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Reto IA: Telefonica challenge:
###Objetivo: entrenar un modelo capaz de distinguir entre un áudio real vs uno generado con IA

## 0. Importación de librerías y datasets

### Git

In [6]:
#Configurar Git
!git config --global user.name "txellbalada"
!git config --global user.email "meritxell.balada@gmail.com"

from google.colab import userdata
# 1. Recuperamos el token de forma segura desde los Secretos de Colab
mi_token = userdata.get('Reto_IA')

# 2. Definimos los datos del repositorio (esto sí puede ser público)
usuario = "txellbalada"
repositorio = "Reto_IA"

# 3. Construimos la URL usando f-strings de Python
repo_url = f"https://{mi_token}@github.com/{usuario}/{repositorio}.git"

# 4. Ejecutamos el comando git clone pasando la variable de Python a Bash usando { }
!git clone {repo_url}

Cloning into 'Reto_IA'...
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 5 (delta 0), reused 2 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (5/5), done.


In [7]:
# Ver el estado de tus archivos
!git status

On branch main

No commits yet

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.config/
	Reto_IA/
	drive/
	sample_data/

nothing added to commit but untracked files present (use "git add" to track)


In [ ]:

# Añadir los cambios
!git add .

# Hacer el commit
!git commit -m "Añadiendo nuevos modelos desde Colab"

# Hacer el push a GitHub
!git push origin main

###Librerías

In [1]:
import numpy as np
import pandas as pd
import librosa
import os
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#Ruta Dani/Sofi
#ls /content/drive/MyDrive/RETO_DE_INTELIGENCIA_ARTIFICIAL/

ls: cannot access '/content/drive/MyDrive/RETO_DE_INTELIGENCIA_ARTIFICIAL/': No such file or directory


### Archivos áudio

In [2]:
#Ruta MBAL
ls /content/drive/MyDrive/00_Master/TFM/Reto_Telefonica/Dataset/

SyntaxError: invalid decimal literal (853296184.py, line 2)

In [ ]:
ruta_carpeta = "/content/drive/MyDrive/00_Master/TFM/Reto_Telefonica/Dataset/"

#Revisamsos qué archivos detecta
archivos = os.listdir(ruta_carpeta)
print(archivos)

#Cargamos los arcihvos - para pruebas , depsués ya lo cambiaremos

for archivo in os.listdir(ruta_carpeta):
    if archivo.endswith(".wav"):
        ruta_audio = os.path.join(ruta_carpeta, archivo)

        y, sr = librosa.load(ruta_audio, sr=16000)

        print(f"{archivo} cargado | duración: {len(y)/sr:.2f} segundos")



['cof_00610_00083325222.wav', 'cof_00610_00106068443.wav', 'cof_00610_00100787111.wav', 'cof_00610_00026180919.wav', 'cof_00610_00008989777.wav', 'cof_00610_00131238701.wav', 'cof_00610_00132949807.wav', 'cof_00610_00011481761.wav', 'cof_00610_00128829414.wav', 'cof_00610_00129157870.wav']
Procesando: /content/drive/MyDrive/00_Master/TFM/Reto_Telefonica/Dataset/cof_00610_00083325222.wav
Procesando: /content/drive/MyDrive/00_Master/TFM/Reto_Telefonica/Dataset/cof_00610_00106068443.wav
Procesando: /content/drive/MyDrive/00_Master/TFM/Reto_Telefonica/Dataset/cof_00610_00100787111.wav
Procesando: /content/drive/MyDrive/00_Master/TFM/Reto_Telefonica/Dataset/cof_00610_00026180919.wav
Procesando: /content/drive/MyDrive/00_Master/TFM/Reto_Telefonica/Dataset/cof_00610_00008989777.wav
Procesando: /content/drive/MyDrive/00_Master/TFM/Reto_Telefonica/Dataset/cof_00610_00131238701.wav
Procesando: /content/drive/MyDrive/00_Master/TFM/Reto_Telefonica/Dataset/cof_00610_00132949807.wav
Procesando: /con

In [ ]:
#Creamos un DF que tenfa ID del áudio y una columna conforme si es generado por IA o no

def generar_df_etiquetas(ruta_carpeta):
    """
    Recorre una carpeta, lee los nombres de los archivos .wav y
    genera un DataFrame con el ID del audio y su etiqueta (0=Real, 1=IA).
    """
    print(f"Explorando carpeta: {ruta_carpeta}")
    datos = []

    for archivo in os.listdir(ruta_carpeta):
        if archivo.endswith(".wav"):
            # 1. Quitamos la extensión para quedarnos con el nombre puro
            nombre_audio = archivo.replace(".wav", "")

            # 2. Lógica de clasificación basada en el nombre
            # Si tiene un guion "-", significa que menciona el algoritmo (ej. StarGAN-) -> Es IA
            if "-" in nombre_audio:
                etiqueta = 1  # Audio generado por IA (Spoof)
            else:
                etiqueta = 0  # Audio Real (Bonafide)

            # 3. Guardamos en la lista de datos
            datos.append({
                "id_audio": nombre_audio,
                "generado_IA": etiqueta
            })

    # Convertimos la lista de diccionarios en un DataFrame de Pandas
    df_etiquetas = pd.DataFrame(datos)

    print(f"DataFrame creado con {len(df_etiquetas)} registros.")
    return df_etiquetas



In [ ]:
#Visualiazamos el DF
df_labels = generar_df_etiquetas(ruta_carpeta)

df_labels.head()

Explorando carpeta: /content/drive/MyDrive/00_Master/TFM/Reto_Telefonica/Dataset/
DataFrame creado con 10 registros.


,id_audio,generado_IA
0,cof_00610_00083325222,0
1,cof_00610_00106068443,0
2,cof_00610_00100787111,0
3,cof_00610_00026180919,0
4,cof_00610_00008989777,0


In [ ]:
#Función final V2_
for archivo in os.listdir(ruta_carpeta):
   if archivo.endswith(".wav"):
     ruta_audio = os.path.join(ruta_carpeta, archivo)

        # 1. Cargas el audio
      #  y, sr = librosa.load(ruta_audio, sr=16000)

        # 2. EXTRAES LAS FEATURES (Aquí es donde ocurre la magia)
        # Puedes llamar a la función que definimos antes:
      #  features_del_audio = extraer_features(ruta_audio)

        # 3. Guardas los resultados en una lista general
        # lista_final.append(features_del_audio)

## 1.0 Extracción de features del áudio

In [ ]:
#Función reutilizable ---> falta documentación de qué hace la función
def extraer_features(ruta_audio):
''' Explicación de qué hace la función'''

    print("\n-----------------------------")
    print("Procesando audio:", ruta_audio)

    y, sr = librosa.load(ruta_audio, sr=16000)
    print("✔ Audio cargado")
    print("   - Frecuencia:", sr)
    print("   - Duración:", round(len(y) / sr, 2), "segundos")

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    print("✔ MFCC calculado:", mfcc.shape)

    zcr = librosa.feature.zero_crossing_rate(y)
    print("✔ ZCR:", zcr.shape)

    rms = librosa.feature.rms(y=y)
    print("✔ RMS:", rms.shape)

    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    print("✔ Spectral Centroid:", centroid.shape)

    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)
    print("✔ Spectral Bandwidth:", bandwidth.shape)

    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
    print("✔ Rolloff:", rolloff.shape)

    features = []
    features.extend(np.mean(mfcc, axis=1))
    features.extend(np.std(mfcc, axis=1))

    features.append(np.mean(zcr))
    features.append(np.std(zcr))

    features.append(np.mean(rms))
    features.append(np.std(rms))

    features.append(np.mean(centroid))
    features.append(np.std(centroid))

    features.append(np.mean(bandwidth))
    features.append(np.std(bandwidth))

    features.append(np.mean(rolloff))
    features.append(np.std(rolloff))

    print("✔ Total features generadas:", len(features))
    print("Primeras 5 features:", features[:5])

    return features

In [ ]:
def extraer_features_V2(ruta_audio):
    """
    Extrae un vector de características (features) de un archivo de audio.
    Calcula medias y desviaciones estándar de: MFCCs, ZCR, RMS y métricas espectrales.

    Argumentos:
        ruta_audio (str): Ruta al archivo .wav o .mp3
    Retorna:
        list: Vector de características listo para el modelo de ML.
    """
    print("\n" + "-"*30)
    print(f"Procesando: {ruta_audio}")

    # 1. Carga y limpieza
    y, sr = librosa.load(ruta_audio, sr=16000)
    y, _ = librosa.effects.trim(y) # Elimina silencios innecesarios

    # 2. Extracción de características base
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    zcr = librosa.feature.zero_crossing_rate(y)
    rms = librosa.feature.rms(y=y)
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)

    # 3. Empaquetado estadístico (Agregamos todo a una lista)
    features = []

    # MFCC (13 medias + 13 desviaciones = 26 features)
    features.extend(np.mean(mfcc, axis=1))
    features.extend(np.std(mfcc, axis=1))

    # Métricas de energía y forma (Usamos listas para iterar y no repetir código)
    metricas = [zcr, rms, centroid, bandwidth, rolloff]
    for m in metricas:
        features.append(np.mean(m))
        features.append(np.std(m))

    print(f"✔ Procesado con éxito. Vector final: {len(features)} dimensiones.")
    return features

Features que obtenemos de cada áudio:

*  MFCC (Coeficientes Cepstrales): Representan el "timbre" o la forma del tracto vocal. Son clave para detectar si la textura de la voz tiene la riqueza de una garganta humana o si presenta los patrones matemáticos repetitivos típicos de los algoritmos de síntesis.

*  ZCR (Zero Crossing Rate): Mide la velocidad a la que la señal cambia de signo. Es fundamental para analizar las consonantes y el ruido de fondo; una IA suele generar transiciones demasiado limpias o ruidos artificiales en frecuencias altas que esta métrica detecta.

*  RMS (Energía): Indica la potencia o el volumen promedio del audio. Se utiliza para identificar si la amplitud es "demasiado plana", ya que los humanos tenemos variaciones de intensidad naturales (énfasis, respiración) que a los modelos de IA les cuesta replicar con realismo.

*  Spectral Centroid (Centroide Espectral): Indica dónde se encuentra el "centro de gravedad" del sonido, definiendo si una voz es "brillante" (aguda) u "opaca" (grave). Ayuda a detectar ese tono "metálico" que suelen tener algunos audios generados artificialmente.

*  Spectral Bandwidth (Ancho de Banda): Mide la extensión o dispersión de las frecuencias. Permite diferenciar la "riqueza" espectral de una grabación real frente a un audio sintético que podría estar limitado a un rango de frecuencias más estrecho o comprimido.

*   Spectral Rolloff: Determina la frecuencia por debajo de la cual se concentra la mayor parte de la energía (85%). Es muy útil para identificar cortes artificiales en el espectro o filtros digitales que se aplican durante el proceso de generación de la IA.

In [ ]:
datos = []
nombres_archivos = []

for archivo in os.listdir(ruta_carpeta):
    if archivo.endswith(".wav"):
        ruta_audio = os.path.join(ruta_carpeta, archivo)
        features = extraer_features(ruta_audio)
        datos.append(features)
        nombres_archivos.append(archivo)

In [ ]:
columnas = []

for i in range(13):
    columnas.append(f"mfcc_{i+1}_mean")

for i in range(13):
    columnas.append(f"mfcc_{i+1}_std")

columnas += [
    "zcr_mean", "zcr_std",
    "rms_mean", "rms_std",
    "centroid_mean", "centroid_std",
    "bandwidth_mean", "bandwidth_std",
    "rolloff_mean", "rolloff_std"
]

print("Número de columnas:", len(columnas))
print(columnas)

In [ ]:
df = pd.DataFrame(datos, columns=columnas)
df["archivo"] = nombres_archivos

df.head()

In [ ]:
df = df[["archivo"] + columnas]
df.head()

In [ ]:
df.to_csv("dataset_features.csv", index=False)
print("✔ Dataset guardado como dataset_features.csv")
print("Shape:", df.shape)

In [ ]:
df.iloc[0]

## 2.0 Modelo NN - > USAR CROSS VALIDATION i adversarial training

In [ ]:
https://github.com/Trusted-AI/adversarial-robustness-toolbox/blob/main/notebooks/classifier_scikitlearn_pipeline_pca_cv_svc.ipynb